# FairTox on Kaggle Notebooks

Backup compute path. Use it if the A100 window has not opened, or if it slips.

**Before this notebook will work:**

1. **Accept the competition rules.** Open the competition, click *Join
   Competition*, accept. Nothing below can see the data until you do.
2. **Add the data.** Right panel -> *Add Input* -> search
   `jigsaw-unintended-bias-in-toxicity-classification` -> add the **Competition**.
3. **Turn on the GPU.** Right panel -> *Session options* -> Accelerator -> `GPU T4 x2`
   or `GPU P100`.
4. **Turn on the internet.** Right panel -> *Session options* -> Internet -> On.
   (Needs a phone-verified account. Without it `git clone` and `pip` fail.)

**Session limits:** 9 hours per session, 30 GPU-hours per week. Anything under
`/kaggle/working` is lost when the session ends unless you **Save Version**.

**The rule that matters scientifically:** runs compared directly must happen on
the same machine in the same session. Keep the baseline/mitigated pair together,
and keep the ablation arms together. A T4 baseline against an A100 mitigated arm
puts a hardware confound inside the comparison the whole study rests on.

In [ ]:
# 1. Confirm the GPU and the data mount before spending quota on setup.
!nvidia-smi --query-gpu=name,memory.total --format=csv
!ls /kaggle/input/jigsaw-unintended-bias-in-toxicity-classification/ 2>/dev/null   || echo 'DATA NOT ATTACHED -- do step 2 in the header above'

In [ ]:
# 2. Get the code.
%cd /kaggle/working
!rm -rf fairtox
!git clone --depth 1 https://github.com/<team>/fairtox.git
%cd /kaggle/working/fairtox

# Kaggle already ships torch, transformers, sklearn, pandas and matplotlib, and
# reinstalling torch from requirements.txt is a reliable way to break the CUDA
# build. Install only what is genuinely missing.
!pip install -q captum tabulate
!python -c "import torch, transformers; print('torch', torch.__version__, '| cuda', torch.cuda.is_available()); print('transformers', transformers.__version__)"

In [ ]:
# 3. Prove the pipeline on synthetic data first (~3 min). If this fails,
# nothing below is worth running.
!python scripts/make_synthetic_data.py --rows 6000
!python -m pytest tests/ -q
!bash run_all.sh configs/smoke.yaml

In [ ]:
# 4. Point at the real competition data and profile it.
#
# /kaggle/input is read-only, but an absolute data.raw_dir overrides the repo
# root, so the pipeline reads the mount in place -- no copying, no disk used.
#
# READ THE OUTPUT. The line that matters is how many subgroups clear the support
# floor ON THE TEST SPLIT: that is how many groups the paper can make a claim
# about. If it warns that too few survive, fix it before training, not after.
!python scripts/run_all.py -c configs/a100.yaml --only eda --set data.raw_dir=/kaggle/input/jigsaw-unintended-bias-in-toxicity-classification

In [ ]:
# 5. Time one epoch on a subset, so the run is sized against a real number
# rather than a guess. Multiply by (full annotated rows / 20000) x epochs.
# run.name=timing keeps this throwaway run from overwriting anything real.
!python scripts/run_all.py -c configs/a100.yaml --only train_baseline --set data.raw_dir=/kaggle/input/jigsaw-unintended-bias-in-toxicity-classification   --set data.subsample=20000   --set training.num_epochs=1   --set run.name=timing

---
## The headline pair

These two cells are the paper's central claim. Run them **back to back, in this
session, with nothing else in between.**

Watch the first 100 steps. If the loss is not falling, stop and diagnose -- it
will not start falling at step 10,000.

In [ ]:
DATA = "--set data.raw_dir=/kaggle/input/jigsaw-unintended-bias-in-toxicity-classification"

!python scripts/run_all.py -c configs/a100.yaml --only train_baseline {DATA}

In [ ]:
!python scripts/run_all.py -c configs/a100.yaml --only train_mitigated {DATA}

In [ ]:
# 6. Audit and compare. No GPU work at all -- this reads saved predictions.
#
# --no-deps is REQUIRED here. Without it the driver resolves compare's
# dependencies and queues both training stages. They would be skipped as
# already-complete, but rely on the flag rather than on the safety net.
#
# Check the verdict line. INVALID means a model collapsed to predicting one
# class: it has perfect FPR parity because it flags nothing. Lower
# mitigation.alpha, re-run the mitigated cell, and do not report it.
!python scripts/run_all.py -c configs/a100.yaml --only compare --no-deps {DATA}

In [ ]:
# 7. Peak VRAM and wall-clock, for the paper's Experimental Setup section.
import json
import pathlib

for model in ("baseline", "mitigated"):
    path = pathlib.Path(f"results/main/metrics/model_{model}.json")
    if not path.exists():
        print(f"{model}: not run yet")
        continue
    d = json.loads(path.read_text())
    t = d["training"]
    print(f"{model:10s} {t['peak_allocated_gb']} GB allocated / {t['peak_reserved_gb']} GB "
          f"reserved | {t['train_seconds']}s | {d['device'].get('gpu_name')}")

In [ ]:
# 8. The ablation (Experiment 5). The long one. Arms whose weights come out
# uniform reuse the baseline instead of retraining it, so this is fewer runs
# than the arm count suggests. Keep the arms together in one session.
!python scripts/run_all.py -c configs/a100.yaml --only ablate --no-deps {DATA}

In [ ]:
# 9. The optional tier. Costs no training: these re-read saved probabilities.
# Threshold sensitivity asks whether the disparity survives a different tau; the
# Jigsaw metrics make the numbers comparable to published work. Cheapest
# quality-per-minute in the project.
#
# Named explicitly with --no-deps. `--tier optional` on its own would resolve
# every MUST stage as well, and that DOES train.
!python scripts/run_all.py -c configs/a100.yaml --no-deps --keep-going {DATA}   --only threshold_sensitivity jigsaw_bias_metrics attributions

In [ ]:
# 10. Collect results somewhere that survives the session.
#
# /kaggle/working is wiped when the session ends unless you SAVE VERSION
# (top right). Do that before closing, then download the output and commit
# results/ to the repo -- the paper cites these files.
!python scripts/freeze_env.py
!mkdir -p /kaggle/working/fairtox_results
!cp -r results/main /kaggle/working/fairtox_results/ 2>/dev/null
!cp requirements.lock.txt /kaggle/working/fairtox_results/ 2>/dev/null
!cd /kaggle/working && tar czf fairtox_results.tar.gz fairtox_results
!ls -la /kaggle/working/fairtox_results.tar.gz
print("\nNow click SAVE VERSION (top right), or this is lost.")